# CUDA Static Cluster-Pool FFN Training Notebook

This notebook is the CUDA-side training gate for the locked FFN-v1 primitive:

```text
static cluster-pool sparse FFN
C = 16 clusters
M = 192 candidate neurons per cluster
dense attention kept on
no MacroV2
```

It follows the same Colab/VSCode pattern as the kernel benchmark notebook: the remote runtime finds or clones the repo, installs it editable, checks that the required CLI commands exist, then runs the training/eval commands from the repo code.

No old checkpoints are assumed to exist in GitHub. By default this notebook trains a dense checkpoint on the remote runtime first. To use an existing remote checkpoint, set `DENSE_CHECKPOINT=/path/on/colab/checkpoint.pt` before running and set `RUN_DENSE_TRAIN=False` in the settings cell.

In [ ]:
from __future__ import annotations

import json
import math
import os
import subprocess
import sys
import time
from pathlib import Path
from typing import Any


def _run(cmd: list[str], *, cwd: Path | None = None) -> None:
    print("$", " ".join(str(x) for x in cmd), flush=True)
    proc = subprocess.Popen(
        [str(x) for x in cmd],
        cwd=str(cwd) if cwd else None,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    tail: list[str] = []
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="", flush=True)
        tail.append(line)
        if len(tail) > 200:
            tail.pop(0)
    rc = proc.wait()
    if rc != 0:
        raise RuntimeError(
            "Command failed with exit code "
            + str(rc)
            + ": "
            + " ".join(str(x) for x in cmd)
            + "\n\nLast command output:\n"
            + "".join(tail)
        )


def _capture(cmd: list[str], *, cwd: Path | None = None) -> str:
    print("$", " ".join(str(x) for x in cmd), flush=True)
    return subprocess.check_output(
        [str(x) for x in cmd],
        cwd=str(cwd) if cwd else None,
        stderr=subprocess.STDOUT,
        text=True,
    )


def _checkout_repo_ref(repo: Path, repo_ref: str) -> Path:
    if not (repo / ".git").exists():
        return repo.resolve()
    if os.environ.get("SPEEDUP_THINGY_UPDATE", "1").strip().lower() in {"0", "false", "no", "off"}:
        return repo.resolve()
    _run(["git", "fetch", "origin"], cwd=repo)
    _run(["git", "checkout", repo_ref], cwd=repo)
    try:
        _run(["git", "pull", "--ff-only"], cwd=repo)
    except Exception:
        print("git pull skipped; likely checked out a detached commit/ref", flush=True)
    return repo.resolve()


def find_or_clone_repo() -> Path:
    repo_ref = os.environ.get("SPEEDUP_THINGY_REF", "master")

    # If the user explicitly points at a checkout, trust that checkout. This is useful for
    # mounted workspaces or a manually prepared Colab folder.
    env_root = os.environ.get("SPEEDUP_THINGY_REPO")
    if env_root:
        candidate = Path(env_root).expanduser()
        if (candidate / "src" / "recursive_training_engine").exists():
            return candidate.resolve()

    # If the notebook is already running from a repo checkout, use it without mutating it.
    cwd = Path.cwd()
    for candidate in (cwd, cwd.parent):
        if (candidate / "src" / "recursive_training_engine").exists():
            return candidate.resolve()

    clone_dir = Path(os.environ.get("SPEEDUP_THINGY_CLONE_DIR", "/content/Speedup-Thingy"))
    repo_url = os.environ.get("SPEEDUP_THINGY_REPO_URL", "https://github.com/BrownHujay/Speedup-Thingy.git")
    clone_dir.parent.mkdir(parents=True, exist_ok=True)
    if not clone_dir.exists():
        try:
            _run(["git", "clone", "--depth", "1", "--branch", repo_ref, repo_url, str(clone_dir)])
        except Exception:
            _run(["git", "clone", repo_url, str(clone_dir)])
            _run(["git", "checkout", repo_ref], cwd=clone_dir)
    elif (clone_dir / "src" / "recursive_training_engine").exists():
        return _checkout_repo_ref(clone_dir, repo_ref)
    else:
        raise RuntimeError(f"Clone directory exists but is not a Speedup-Thingy repo: {clone_dir}")
    return clone_dir.resolve()


repo_root = find_or_clone_repo()
src_path = repo_root / "src"
os.environ["PYTHONPATH"] = str(src_path) + os.pathsep + os.environ.get("PYTHONPATH", "")
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

try:
    import yaml  # noqa: F401
except ModuleNotFoundError:
    _run([sys.executable, "-m", "pip", "install", "-q", "pyyaml"])

try:
    import recursive_training_engine  # noqa: F401
except ModuleNotFoundError:
    _run([sys.executable, "-m", "pip", "install", "-q", "-e", str(repo_root), "--no-deps"])

import torch

if not torch.cuda.is_available():
    raise RuntimeError("Select a CUDA Colab runtime before running this notebook.")

torch.backends.cuda.matmul.allow_tf32 = True


def rte_cmd(*args: object) -> list[str]:
    return [sys.executable, "-c", "from recursive_training_engine.cli import main; main()", *[str(a) for a in args]]


def run_rte(*args: object) -> None:
    _run(rte_cmd(*args), cwd=repo_root)


help_text = _capture(rte_cmd("--help"), cwd=repo_root)
required_commands = [
    "train",
    "static-cluster-pool-continuation",
    "static-cluster-pool-gradient-alignment",
    "static-cluster-pool-union-eval",
    "benchmark-static-cluster-pool-ffn-train",
    "benchmark-active-union-ffn-train",
    "benchmark-active-union-model-train-step",
]
missing = [name for name in required_commands if name not in help_text]
if missing:
    raise RuntimeError(
        "This remote checkout does not contain the static cluster-pool training commands: "
        + ", ".join(missing)
        + "\nSet SPEEDUP_THINGY_REF to the branch/commit that has the latest kernels/CLI."
    )

print("repo_root:", repo_root)
print("torch:", torch.__version__)
print("cuda device:", torch.cuda.get_device_name(0))
print("capability:", torch.cuda.get_device_capability(0))


## Settings

Defaults are deliberately CUDA-Colab friendly: `d_model=256`, `d_ff=2048`, synthetic tokens, 400 dense pretrain steps, then the 50/300-step continuation gates. Raise `D_MODEL` to `512` after the first pass if you want the larger-H validation from the guide.

The notebook writes every artifact under a timestamped remote `runs/colab_static_cluster_ffn_*` directory.

In [ ]:
def _bool_env(name: str, default: bool) -> bool:
    raw = os.environ.get(name)
    if raw is None:
        return default
    return raw.strip().lower() in {"1", "true", "yes", "y", "on"}


def _int_list_env(name: str, default: list[int]) -> list[int]:
    raw = os.environ.get(name)
    if raw is None or not raw.strip():
        return list(default)
    return [int(x.strip()) for x in raw.split(",") if x.strip()]


# Larger-H quality/training validation.
D_MODEL = int(os.environ.get("D_MODEL", "256"))
D_FF = int(os.environ.get("D_FF", "2048"))
N_LAYERS = int(os.environ.get("N_LAYERS", "6"))
N_HEADS = int(os.environ.get("N_HEADS", "8"))
VOCAB_SIZE = int(os.environ.get("VOCAB_SIZE", "2048"))
SEQ_LEN = int(os.environ.get("SEQ_LEN", "64"))
BATCH_SIZE = int(os.environ.get("BATCH_SIZE", "32"))
PRECISION = os.environ.get("PRECISION", "fp32")
SEED = int(os.environ.get("SEED", "1234"))

# FFN-v1 primitive locked by the previous gates.
CLUSTERS = int(os.environ.get("CLUSTERS", "16"))
CANDIDATE_M = int(os.environ.get("CANDIDATE_M", "192"))
SVD_RANK = int(os.environ.get("SVD_RANK", "64"))
CALIBRATION_TOKENS = int(os.environ.get("CALIBRATION_TOKENS", "8192"))
CLUSTER_ITERS = int(os.environ.get("CLUSTER_ITERS", "4"))

# Dense pretrain and continuation gates.
DENSE_CKPT_STEPS = _int_list_env("DENSE_CKPT_STEPS", [100, 200, 400])
CONT_LR = float(os.environ.get("CONT_LR", "1e-4"))
CONT_WEIGHT_DECAY = float(os.environ.get("CONT_WEIGHT_DECAY", "0.0"))
EVAL_TOKENS = int(os.environ.get("EVAL_TOKENS", "65536"))
EVAL_BATCHES = max(1, math.ceil(EVAL_TOKENS / (BATCH_SIZE * SEQ_LEN)))
ALIGNMENT_BATCHES = int(os.environ.get("ALIGNMENT_BATCHES", "1"))
ALIGNMENT_BATCH_SIZE = int(os.environ.get("ALIGNMENT_BATCH_SIZE", str(BATCH_SIZE)))

RUN_DENSE_TRAIN = _bool_env("RUN_DENSE_TRAIN", True)
RUN_50_CONTINUATION = _bool_env("RUN_50_CONTINUATION", True)
RUN_300_CONTINUATION = _bool_env("RUN_300_CONTINUATION", True)
RUN_FWD_BWD_BENCH = _bool_env("RUN_FWD_BWD_BENCH", True)
RUN_UNION_EVAL = _bool_env("RUN_UNION_EVAL", True)
RUN_UNION_CAP_EVAL = _bool_env("RUN_UNION_CAP_EVAL", True)
RUN_EARLY_UNION_EVAL = _bool_env("RUN_EARLY_UNION_EVAL", False)
RUN_ACTIVE_UNION_BENCH = _bool_env("RUN_ACTIVE_UNION_BENCH", True)
RUN_ACTIVE_UNION_CONTINUATION = _bool_env("RUN_ACTIVE_UNION_CONTINUATION", True)
RUN_FULL_MODEL_SPEED_BENCH = _bool_env("RUN_FULL_MODEL_SPEED_BENCH", True)
RUN_EARLY_HANDOFF = _bool_env("RUN_EARLY_HANDOFF", True)
RESUME_EXISTING = _bool_env("RESUME_EXISTING", True)

# Optional data switch. Synthetic is the default so this runs without dataset downloads.
USE_TINYSTORIES = _bool_env("USE_TINYSTORIES", False)
SYNTHETIC_TOKENS = int(os.environ.get("SYNTHETIC_TOKENS", "2000000"))

# If set, this remote checkpoint is used instead of training a dense checkpoint here.
DENSE_CHECKPOINT_OVERRIDE = os.environ.get("DENSE_CHECKPOINT", "").strip()

RUN_STAMP = os.environ.get("RUN_STAMP", time.strftime("%Y%m%d_%H%M%S"))
RUN_ROOT = repo_root / "runs" / f"colab_static_cluster_ffn_d{D_MODEL}_h{D_FF}_{RUN_STAMP}"
CHECKPOINT_DIR = RUN_ROOT / "checkpoints"
REPORT_DIR = RUN_ROOT / "reports"
CONFIG_DIR = RUN_ROOT / "configs"
for path in (CHECKPOINT_DIR, REPORT_DIR, CONFIG_DIR):
    path.mkdir(parents=True, exist_ok=True)

BENCH_SIZES = os.environ.get("BENCH_SIZES", "512x2048x1024,2048x8192x1024")
BENCH_SIZES = [x.strip() for x in BENCH_SIZES.split(",") if x.strip()]
ACTIVE_UNION_BENCH_SIZES = os.environ.get("ACTIVE_UNION_BENCH_SIZES", ",".join(BENCH_SIZES))
ACTIVE_UNION_BENCH_SIZES = [x.strip() for x in ACTIVE_UNION_BENCH_SIZES.split(",") if x.strip()]
ACTIVE_UNION_MS = _int_list_env("ACTIVE_UNION_MS", [160, 192, 256, 320])
BENCH_ITERS = int(os.environ.get("BENCH_ITERS", "5"))
BENCH_WARMUP = int(os.environ.get("BENCH_WARMUP", "1"))
# 0 means uncapped active union. These are quality-changing variants, so keep the flags explicit.
ACTIVE_UNION_CONT_CAPS = _int_list_env("ACTIVE_UNION_CONT_CAPS", [0, 320, 256, 192])
FULL_MODEL_BENCH_CAPS = _int_list_env("FULL_MODEL_BENCH_CAPS", [0, 320, 256, 192])
ACTIVE_UNION_HANDOFF_STEP = int(os.environ.get("ACTIVE_UNION_HANDOFF_STEP", "200"))
ACTIVE_UNION_CONT_STEPS = int(os.environ.get("ACTIVE_UNION_CONT_STEPS", "300"))
FULL_MODEL_BENCH_DTYPE = os.environ.get("FULL_MODEL_BENCH_DTYPE", "fp16")
FULL_MODEL_BENCH_ITERS = int(os.environ.get("FULL_MODEL_BENCH_ITERS", str(BENCH_ITERS)))
FULL_MODEL_BENCH_WARMUP = int(os.environ.get("FULL_MODEL_BENCH_WARMUP", str(BENCH_WARMUP)))
FULL_MODEL_COMPONENT_ITERS = int(os.environ.get("FULL_MODEL_COMPONENT_ITERS", "3"))
FULL_MODEL_COMPONENT_WARMUP = int(os.environ.get("FULL_MODEL_COMPONENT_WARMUP", "1"))
UNION_CAPS = _int_list_env("UNION_CAPS", [160, 192, 256, 320])
# Optional per-layer caps, e.g. UNION_LAYER_CAPS=192,192,256,256,320,320 for 6 layers.
# Empty by default because this is a quality-changing experiment.
UNION_LAYER_CAPS = _int_list_env("UNION_LAYER_CAPS", [])
RUN_CUDA_GRAPHS = _bool_env("RUN_CUDA_GRAPHS", True)
CUDA_GRAPH_WARMUP = int(os.environ.get("CUDA_GRAPH_WARMUP", "3"))
RUN_TRITON_SWIGLU_BACKWARD = _bool_env("RUN_TRITON_SWIGLU_BACKWARD", True)
EARLY_HANDOFF_STEPS = _int_list_env("EARLY_HANDOFF_STEPS", [100, 200])
EARLY_HANDOFF_CONT_STEPS = int(os.environ.get("EARLY_HANDOFF_CONT_STEPS", "300"))

settings = {
    "run_root": str(RUN_ROOT),
    "model": {"d_model": D_MODEL, "d_ff": D_FF, "layers": N_LAYERS, "heads": N_HEADS, "vocab": VOCAB_SIZE},
    "training": {"batch_size": BATCH_SIZE, "seq_len": SEQ_LEN, "precision": PRECISION, "seed": SEED},
    "ffn_v1": {"clusters": CLUSTERS, "candidate_m": CANDIDATE_M, "rank": SVD_RANK, "calibration_tokens": CALIBRATION_TOKENS},
    "eval_batches": EVAL_BATCHES,
    "dense_ckpt_steps": DENSE_CKPT_STEPS,
    "toggles": {
        "run_dense_train": RUN_DENSE_TRAIN,
        "run_50_continuation": RUN_50_CONTINUATION,
        "run_300_continuation": RUN_300_CONTINUATION,
        "run_fwd_bwd_bench": RUN_FWD_BWD_BENCH,
        "run_union_eval": RUN_UNION_EVAL,
        "run_union_cap_eval": RUN_UNION_CAP_EVAL,
        "union_caps": UNION_CAPS if RUN_UNION_CAP_EVAL else [],
        "union_layer_caps": UNION_LAYER_CAPS if RUN_UNION_CAP_EVAL else [],
        "run_early_union_eval": RUN_EARLY_UNION_EVAL,
        "run_active_union_bench": RUN_ACTIVE_UNION_BENCH,
        "active_union_ms": ACTIVE_UNION_MS,
        "run_active_union_continuation": RUN_ACTIVE_UNION_CONTINUATION,
        "active_union_cont_caps": ACTIVE_UNION_CONT_CAPS,
        "active_union_handoff_step": ACTIVE_UNION_HANDOFF_STEP,
        "active_union_cont_steps": ACTIVE_UNION_CONT_STEPS,
        "run_full_model_speed_bench": RUN_FULL_MODEL_SPEED_BENCH,
        "full_model_bench_caps": FULL_MODEL_BENCH_CAPS,
        "full_model_bench_dtype": FULL_MODEL_BENCH_DTYPE,
        "run_cuda_graphs": RUN_CUDA_GRAPHS,
        "cuda_graph_warmup": CUDA_GRAPH_WARMUP,
        "run_triton_swiglu_backward": RUN_TRITON_SWIGLU_BACKWARD,
        "run_early_handoff": RUN_EARLY_HANDOFF,
    },
}
print(json.dumps(settings, indent=2))

if USE_TINYSTORIES:
    _run([sys.executable, "-m", "pip", "install", "-q", "tokenizers", "datasets"])


## Generate The Config

This writes a remote config under the run directory. The file is JSON with a `.yaml` suffix; JSON is valid YAML, and this avoids notebook-side YAML dependencies.

In [ ]:
def _divisor_at_most(value: int, limit: int) -> int:
    candidate = min(value, limit)
    while candidate > 1 and value % candidate != 0:
        candidate -= 1
    return max(1, candidate)


ffn_groups = _divisor_at_most(D_FF, 64)
head_groups = _divisor_at_most(N_HEADS, N_HEADS)
config = {
    "run_name": f"colab_static_cluster_ffn_d{D_MODEL}_h{D_FF}",
    "output_dir": str(RUN_ROOT),
    "model": {
        "topology": "dense",
        "vocab_size": VOCAB_SIZE,
        "d_model": D_MODEL,
        "n_heads": N_HEADS,
        "d_ff": D_FF,
        "n_dense_layers": N_LAYERS,
        "n_prelude": 1,
        "n_coda": 1,
        "t_max": 4,
        "depth_choices": [1, 2, 4],
        "attn_banks": 1,
        "ffn_banks": 2,
        "head_groups": head_groups,
        "ffn_groups": ffn_groups,
        "active_head_groups": min(2, head_groups),
        "active_ffn_groups": min(4, ffn_groups),
        "recipe_count": 16,
        "tie_embeddings": True,
        "use_rope": True,
        "use_recursive_input_skip": True,
        "fairness_tolerance": 0.01,
        "macro_rank": max(1, min(32, D_MODEL // 2)),
        "macro_hidden_mult": 2,
        "macro_use_gated_update": True,
        "macro_update_scale": 0.05,
        "macro_update_scale_init": 0.1,
        "macro_include_delta_to_h0": True,
        "macro_use_depth_embedding": True,
        "macro_decomposition": "direct",
        "router_hidden": max(64, D_MODEL // 2),
    },
    "training": {
        "mode": "dense_exact",
        "optimizer": "adamw",
        "lr": 1e-3,
        "weight_decay": 0.1,
        "batch_size": BATCH_SIZE,
        "seq_len": SEQ_LEN,
        "grad_accum_steps": 1,
        "grad_clip_norm": 1.0,
        "seed": SEED,
        "precision": PRECISION,
        "audit_p_min": 0.0,
        "audit_p_max": 0.0,
        "audit_alpha": 0.0,
        "audit_beta": 0.0,
        "audit_gamma": 0.0,
        "audit_mode": "metric_only",
        "audit_gradient_correction": False,
        "lambda_cons": 0.001,
        "disable_router_aux": True,
        "debug_force_full_output": True,
        "coverage_min": 0.0,
        "log_every": 25,
        "compile_model": False,
        "fused_optimizer": False,
        "foreach_optimizer": False,
        "allow_tf32": True,
        "strict_cuda": True,
        "require_triton": False,
        "require_flash_attention": False,
    },
    "output": {"mode": "full"},
    "data": {
        "dataset": "tinystories" if USE_TINYSTORIES else "synthetic",
        "tokenizer": "gpt2_bpe",
        "vocab_projection": "filter",
        "projection_lane": "filter",
        "train_tokens": SYNTHETIC_TOKENS if USE_TINYSTORIES else None,
        "max_tokens": SYNTHETIC_TOKENS,
        "eval_tokens": EVAL_TOKENS,
        "synthetic_tokens": SYNTHETIC_TOKENS,
        "seed": SEED,
    },
}
# Drop None values so the dataclass defaults behave normally.
config["data"] = {k: v for k, v in config["data"].items() if v is not None}

CONFIG_PATH = CONFIG_DIR / f"static_cluster_ffn_d{D_MODEL}_h{D_FF}.yaml"
CONFIG_PATH.write_text(json.dumps(config, indent=2))
print("CONFIG_PATH:", CONFIG_PATH)
print(CONFIG_PATH.read_text()[:2000])

## Train Or Load Dense Checkpoints

This is chunked so early handoff checkpoints at steps `100` and `200` exist. Each checkpoint includes optimizer state, which matters for clean continuation.

In [ ]:
dense_checkpoints: dict[int | str, Path] = {}


def train_dense_checkpoints_in_process() -> dict[int, Path]:
    import dataclasses

    from recursive_training_engine.cli import _model_for_mode, _with_run_dir
    from recursive_training_engine.config import load_config
    from recursive_training_engine.data import load_token_streams
    from recursive_training_engine.training import TrainEngine
    from recursive_training_engine.utils import set_seed

    cfg = _with_run_dir(
        _model_for_mode(load_config(str(CONFIG_PATH)), "dense_exact"),
        str(RUN_ROOT / "dense_train_direct"),
    )
    set_seed(cfg.training.seed)
    streams = load_token_streams(cfg.data, cfg.training, cfg.model.vocab_size)
    engine = TrainEngine(cfg)
    engine.write_run_manifest(
        {
            "data_fingerprint": streams.data_fingerprint,
            "tokenizer": streams.tokenizer_name,
            "projection_lane": cfg.data.vocab_projection,
            "vocab_size": cfg.model.vocab_size,
            "seq_len": cfg.training.seq_len,
            "batch_size": cfg.training.batch_size,
            "mode": cfg.training.mode,
            "train_tokens": int(streams.train.numel()),
            "eval_tokens": int(streams.eval.numel()),
            "notebook": "cuda_static_cluster_pool_ffn_train.ipynb",
            "checkpoint_steps": sorted(set(DENSE_CKPT_STEPS)),
        }
    )
    batches = streams.train_batches(cfg.training)
    completed_steps = 0
    checkpoints: dict[int, Path] = {}
    try:
        for target_step in sorted(set(DENSE_CKPT_STEPS)):
            ckpt = CHECKPOINT_DIR / f"dense_step{target_step}.pt"
            checkpoints[target_step] = ckpt
            if RESUME_EXISTING and ckpt.exists():
                # Verify it is readable before trusting it. If it is corrupt, retrain it.
                try:
                    import torch

                    torch.load(ckpt, map_location="cpu", weights_only=False)
                    print(f"Reusing existing checkpoint for step {target_step}: {ckpt}")
                    completed_steps = target_step
                    engine.load_checkpoint(ckpt)
                    continue
                except Exception as exc:
                    print(f"Existing checkpoint failed to load, retraining {target_step}: {exc!r}")
                    ckpt.unlink(missing_ok=True)
            while completed_steps < target_step:
                result = engine.train_step(next(batches))
                completed_steps += 1
                if completed_steps == 1 or completed_steps % max(1, cfg.training.log_every) == 0:
                    printable = {
                        key: (float(value.detach().float().cpu()) if hasattr(value, "detach") else value)
                        for key, value in result.metrics.items()
                    }
                    printable["step"] = completed_steps
                    print(json.dumps(printable, sort_keys=True), flush=True)
            engine.save_checkpoint(ckpt)
            print(f"saved dense checkpoint step {target_step}: {ckpt}", flush=True)
    finally:
        engine.close()
    return checkpoints


if DENSE_CHECKPOINT_OVERRIDE and not RUN_DENSE_TRAIN:
    DENSE_CHECKPOINT = Path(DENSE_CHECKPOINT_OVERRIDE).expanduser()
    if not DENSE_CHECKPOINT.exists():
        raise FileNotFoundError(f"DENSE_CHECKPOINT does not exist on this remote runtime: {DENSE_CHECKPOINT}")
    dense_checkpoints["override"] = DENSE_CHECKPOINT
    print("Using provided dense checkpoint:", DENSE_CHECKPOINT)
else:
    dense_checkpoints.update(train_dense_checkpoints_in_process())
    DENSE_CHECKPOINT = dense_checkpoints[max(k for k in dense_checkpoints if isinstance(k, int))]
    if not DENSE_CHECKPOINT.exists():
        raise FileNotFoundError(f"Dense checkpoint was not created: {DENSE_CHECKPOINT}")
    print("Final dense checkpoint:", DENSE_CHECKPOINT)

print("Dense checkpoints:")
for key, value in dense_checkpoints.items():
    print(" ", key, value)


## Static Cluster-Pool Continuation Helpers

The continuation command compares dense continuation against static sparse-FFN continuation with dense attention still enabled. It also logs gradient alignment, coverage, row-gradient norms, and sparse gaps versus dense at each eval step.

In [ ]:
def run_continuation(
    *,
    name: str,
    checkpoint: Path,
    steps: int,
    eval_steps: list[int],
    refresh_intervals: list[int],
    run_dense: bool = True,
    run_sparse: bool = True,
    sparse_ffn_kind: str = "static_cluster",
    active_union_cap: int | None = None,
    include_gradient_alignment: bool = True,
) -> Path:
    output = REPORT_DIR / f"{name}.json"
    if RESUME_EXISTING and output.exists():
        print("Reusing existing report:", output)
        return output
    args: list[object] = [
        "static-cluster-pool-continuation",
        "--config", CONFIG_PATH,
        "--dense-checkpoint", checkpoint,
        "--steps", steps,
        "--lr", CONT_LR,
        "--weight-decay", CONT_WEIGHT_DECAY,
        "--resume-optimizer-state",
        "--eval-batches", EVAL_BATCHES,
        "--eval-steps", *eval_steps,
        "--calibration-tokens", CALIBRATION_TOKENS,
        "--rank", SVD_RANK,
        "--clusters", CLUSTERS,
        "--candidate-m", CANDIDATE_M,
        "--sparse-ffn-kind", sparse_ffn_kind,
        "--refresh-intervals", *refresh_intervals,
        "--cluster-iters", CLUSTER_ITERS,
        "--alignment-batches", ALIGNMENT_BATCHES,
        "--alignment-batch-size", ALIGNMENT_BATCH_SIZE,
        "--output", output,
    ]
    if include_gradient_alignment:
        args.append("--include-gradient-alignment")
    if active_union_cap is not None and active_union_cap > 0:
        args.extend(["--active-union-cap", active_union_cap])
    args.append("--run-dense" if run_dense else "--no-run-dense")
    args.append("--run-sparse" if run_sparse else "--no-run-sparse")
    run_rte(*args)
    return output


continuation_reports: dict[str, Path] = {}
active_union_continuation_reports: dict[str, Path] = {}


## 50-Step Sanity Gate

This is the quick check: dense must be stable, and static sparse should not widen the initial gap.

In [ ]:
if RUN_50_CONTINUATION:
    continuation_reports["final_50"] = run_continuation(
        name="final_checkpoint_50_steps",
        checkpoint=DENSE_CHECKPOINT,
        steps=50,
        eval_steps=[0, 50],
        refresh_intervals=[0, 50],
    )
else:
    print("Skipping 50-step continuation.")

## 300-Step Stability Gate

This is the main gate from the guide: dense continuation, no-refresh static sparse FFN, refresh@100, and refresh@50. Eval is logged at `0, 50, 100, 200, 300` over `65,536` tokens by default.

In [ ]:
if RUN_300_CONTINUATION:
    continuation_reports["final_300"] = run_continuation(
        name="final_checkpoint_300_steps",
        checkpoint=DENSE_CHECKPOINT,
        steps=300,
        eval_steps=[0, 50, 100, 200, 300],
        refresh_intervals=[0, 100, 50],
    )
else:
    print("Skipping 300-step continuation.")

## Forward + Backward CUDA Benchmark

This measures dense FFN versus static C16/M192 FFN forward+backward and includes the scatter cost. The target gate is at least `5x` for `d=2048,H=8192` including scatter, with `10x` as the strong gate.

In [ ]:
if RUN_FWD_BWD_BENCH:
    bench_output = REPORT_DIR / "static_cluster_pool_fwd_bwd_benchmark.json"
    if RESUME_EXISTING and bench_output.exists():
        print("Reusing existing benchmark:", bench_output)
    else:
        args: list[object] = [
            "benchmark-static-cluster-pool-ffn-train",
            "--clusters", CLUSTERS,
            "--candidate-m", CANDIDATE_M,
            "--iters", BENCH_ITERS,
            "--warmup", BENCH_WARMUP,
            "--cuda-graph-warmup", CUDA_GRAPH_WARMUP,
            "--output", bench_output,
        ]
        args.append("--cuda-graphs" if RUN_CUDA_GRAPHS else "--no-cuda-graphs")
        for size in BENCH_SIZES:
            args.extend(["--size", size])
        run_rte(*args)
else:
    bench_output = None
    print("Skipping forward+backward benchmark.")


## Early-Checkpoint Handoff

This tests whether sparse FFN works before the dense model is fully polished. It uses the chunked checkpoints created above, if available. If you supplied an external final checkpoint and did not train dense chunks in this notebook, this section skips cleanly.

In [ ]:
if RUN_EARLY_HANDOFF:
    for handoff_step in EARLY_HANDOFF_STEPS:
        ckpt = dense_checkpoints.get(handoff_step)
        if ckpt is None or not Path(ckpt).exists():
            print(f"Skipping early handoff at step {handoff_step}; checkpoint is unavailable.")
            continue
        continuation_reports[f"early_{handoff_step}_300"] = run_continuation(
            name=f"early_handoff_step{handoff_step}_{EARLY_HANDOFF_CONT_STEPS}_steps",
            checkpoint=Path(ckpt),
            steps=EARLY_HANDOFF_CONT_STEPS,
            eval_steps=[0, 50, 100, 200, EARLY_HANDOFF_CONT_STEPS],
            refresh_intervals=[0],
        )
else:
    print("Skipping early handoff tests.")

## Static Cluster Union Eval

This is the new H=2048 check: build the normal static `C16 M192` codebook, evaluate the clustered path, then evaluate one global active FFN using the unique union of the cluster pools. This answers whether the simpler active-union path preserves quality without cluster routing/packing.

When `RUN_UNION_CAP_EVAL=True`, the command also evaluates capped active sets (`UNION_CAPS`, default `160,192,256,320`). Those caps can change model quality, so the flag exists to disable them cleanly. `UNION_LAYER_CAPS` can also be set for a single per-layer cap variant when we want to test paying more active neurons only in the layers that need it.



In [ ]:
union_eval_reports: dict[str, Path] = {}

if RUN_UNION_EVAL:
    union_output = REPORT_DIR / "static_cluster_union_eval_final.json"
    if RESUME_EXISTING and union_output.exists():
        print("Reusing existing union eval:", union_output)
    else:
        args: list[object] = [
            "static-cluster-pool-union-eval",
            "--config", CONFIG_PATH,
            "--dense-checkpoint", DENSE_CHECKPOINT,
            "--eval-batches", EVAL_BATCHES,
            "--calibration-tokens", CALIBRATION_TOKENS,
            "--rank", SVD_RANK,
            "--clusters", CLUSTERS,
            "--candidate-m", CANDIDATE_M,
            "--cluster-iters", CLUSTER_ITERS,
            "--output", union_output,
        ]
        if RUN_UNION_CAP_EVAL and UNION_CAPS:
            args.extend(["--union-caps", *UNION_CAPS])
        if RUN_UNION_CAP_EVAL and UNION_LAYER_CAPS:
            args.extend(["--union-layer-caps", *UNION_LAYER_CAPS])
        run_rte(*args)
    union_eval_reports["final"] = union_output

    if RUN_EARLY_UNION_EVAL:
        for handoff_step in EARLY_HANDOFF_STEPS:
            ckpt = dense_checkpoints.get(handoff_step)
            if ckpt is None or not Path(ckpt).exists():
                print(f"Skipping union eval at step {handoff_step}; checkpoint is unavailable.")
                continue
            out = REPORT_DIR / f"static_cluster_union_eval_step{handoff_step}.json"
            if RESUME_EXISTING and out.exists():
                print("Reusing existing early union eval:", out)
            else:
                args: list[object] = [
                    "static-cluster-pool-union-eval",
                    "--config", CONFIG_PATH,
                    "--dense-checkpoint", ckpt,
                    "--eval-batches", EVAL_BATCHES,
                    "--calibration-tokens", CALIBRATION_TOKENS,
                    "--rank", SVD_RANK,
                    "--clusters", CLUSTERS,
                    "--candidate-m", CANDIDATE_M,
                    "--cluster-iters", CLUSTER_ITERS,
                    "--output", out,
                ]
                if RUN_UNION_CAP_EVAL and UNION_CAPS:
                    args.extend(["--union-caps", *UNION_CAPS])
                if RUN_UNION_CAP_EVAL and UNION_LAYER_CAPS:
                    args.extend(["--union-layer-caps", *UNION_LAYER_CAPS])
                run_rte(*args)
            union_eval_reports[f"step{handoff_step}"] = out
else:
    print("Skipping static cluster union eval.")


## Active-Union Forward + Backward Benchmark

This is the systems probe for the union idea. It compares dense FFN against two active-union training shapes:

```text
indexed master weights: W_full.index_select(active_ids) each step
packed active weights:  W_active is already contiguous/trainable
packed fused WUG:       W_up/W_gate active rows are concatenated into one W_ug GEMM
```

If packed active is fast and indexed master is slow, the next training systems path is pool/union-native weights with periodic reconcile, not per-step dense-master scatter.

Dense baseline uses fused `W_ug` too, matching the real dense SwiGLU path. This avoids giving sparse an unfair benchmark against a deliberately fragmented dense FFN.


In [ ]:
if RUN_ACTIVE_UNION_BENCH:
    active_union_bench_output = REPORT_DIR / "active_union_fwd_bwd_benchmark.json"
    if RESUME_EXISTING and active_union_bench_output.exists():
        print("Reusing existing active-union benchmark:", active_union_bench_output)
    else:
        args: list[object] = [
            "benchmark-active-union-ffn-train",
            "--active-m", *ACTIVE_UNION_MS,
            "--iters", BENCH_ITERS,
            "--warmup", BENCH_WARMUP,
            "--cuda-graph-warmup", CUDA_GRAPH_WARMUP,
            "--output", active_union_bench_output,
        ]
        args.append("--cuda-graphs" if RUN_CUDA_GRAPHS else "--no-cuda-graphs")
        args.append("--triton-swiglu-backward" if RUN_TRITON_SWIGLU_BACKWARD else "--no-triton-swiglu-backward")
        for size in ACTIVE_UNION_BENCH_SIZES:
            args.extend(["--size", size])
        run_rte(*args)
else:
    active_union_bench_output = None
    print("Skipping active-union benchmark.")




## Active-Union Packed Continuation

This is the FFN-v2 training check. It uses the step-200 handoff checkpoint when available, installs packed active-union FFNs, and runs uncapped/capped active sets. `0` in `ACTIVE_UNION_CONT_CAPS` means uncapped. The sparse branch uses packed active weights directly; it does not train through indexed dense-master rows.


In [ ]:
if RUN_ACTIVE_UNION_CONTINUATION:
    active_union_checkpoint = dense_checkpoints.get(ACTIVE_UNION_HANDOFF_STEP, DENSE_CHECKPOINT)
    if active_union_checkpoint is None or not Path(active_union_checkpoint).exists():
        print(f"Step {ACTIVE_UNION_HANDOFF_STEP} checkpoint unavailable; using final dense checkpoint.")
        active_union_checkpoint = DENSE_CHECKPOINT
    print("Active-union continuation checkpoint:", active_union_checkpoint)
    for idx, cap in enumerate(ACTIVE_UNION_CONT_CAPS):
        cap_label = "uncapped" if int(cap) <= 0 else f"cap{int(cap)}"
        active_union_continuation_reports[cap_label] = run_continuation(
            name=f"active_union_packed_{cap_label}_{ACTIVE_UNION_CONT_STEPS}_steps",
            checkpoint=Path(active_union_checkpoint),
            steps=ACTIVE_UNION_CONT_STEPS,
            eval_steps=sorted({0, 50, 100, 200, ACTIVE_UNION_CONT_STEPS}),
            refresh_intervals=[0],
            run_dense=(idx == 0),
            run_sparse=True,
            sparse_ffn_kind="active_union_packed",
            active_union_cap=None if int(cap) <= 0 else int(cap),
            include_gradient_alignment=False,
        )
else:
    print("Skipping active-union packed continuation.")


## Full-Model Train-Step Speed Benchmark

This compares the real dense model train step against dense-attention + packed active-union FFN. Dense gets a packed row-oriented SwiGLU baseline too, and the Triton SwiGLU backward flag is applied symmetrically to dense packed FFNs and sparse packed FFNs. Sparse-specific speedups stay scoped to replacing the FFN weight shape; the rest of the model is not privately compiled for sparse.


In [ ]:
if RUN_FULL_MODEL_SPEED_BENCH:
    full_model_bench_output = REPORT_DIR / "active_union_full_model_train_step_benchmark.json"
    if RESUME_EXISTING and full_model_bench_output.exists():
        print("Reusing existing full-model speed benchmark:", full_model_bench_output)
    else:
        args: list[object] = [
            "benchmark-active-union-model-train-step",
            "--config", CONFIG_PATH,
            "--dense-checkpoint", DENSE_CHECKPOINT,
            "--active-union-cap", *FULL_MODEL_BENCH_CAPS,
            "--calibration-tokens", CALIBRATION_TOKENS,
            "--rank", SVD_RANK,
            "--clusters", CLUSTERS,
            "--candidate-m", CANDIDATE_M,
            "--cluster-iters", CLUSTER_ITERS,
            "--dtype", FULL_MODEL_BENCH_DTYPE,
            "--lr", CONT_LR,
            "--weight-decay", CONT_WEIGHT_DECAY,
            "--iters", FULL_MODEL_BENCH_ITERS,
            "--warmup", FULL_MODEL_BENCH_WARMUP,
            "--component-iters", FULL_MODEL_COMPONENT_ITERS,
            "--component-warmup", FULL_MODEL_COMPONENT_WARMUP,
            "--output", full_model_bench_output,
        ]
        args.append("--triton-swiglu-backward" if RUN_TRITON_SWIGLU_BACKWARD else "--no-triton-swiglu-backward")
        run_rte(*args)
else:
    full_model_bench_output = None
    print("Skipping full-model train-step speed benchmark.")


## Summary Tables

This cell reads the JSON reports and prints compact pass/fail-facing summaries. The full reports stay in the remote run directory.

In [ ]:
def _load_json(path: Path) -> dict[str, Any]:
    return json.loads(Path(path).read_text())


def summarize_continuation(path: Path) -> None:
    report = _load_json(path)
    print("\n===", path.name, "===")
    rows = report.get("rows", [])
    dense = next((row for row in rows if row.get("variant") == "dense_continuation"), None)
    if dense:
        print(
            f"dense: {dense.get('initial_nll_per_token'):.6f} -> {dense.get('final_nll_per_token'):.6f} "
            f"delta={dense.get('nll_delta'):.6f}"
        )
    for row in rows:
        variant = row.get("variant")
        if variant == "dense_continuation":
            continue
        init = row.get("initial_nll_per_token")
        final = row.get("final_nll_per_token")
        init_gap = row.get("initial_gap_vs_dense")
        final_gap = row.get("final_gap_vs_dense")
        gap_text = (
            f"gap {init_gap:.6f} -> {final_gap:.6f}"
            if init_gap is not None and final_gap is not None
            else "gap unavailable in this report"
        )
        print(f"{variant}: {init:.6f} -> {final:.6f} {gap_text}")
        align_initial = row.get("initial_gradient_alignment") or {}
        align_final = row.get("final_gradient_alignment") or {}
        if align_initial:
            print(
                "  grad cos initial/final: input="
                f"{align_initial.get('input_grad_cosine')} / {align_final.get('input_grad_cosine')}"
            )
        coverage = (row.get("final_coverage") or {}).get("mean") or {}
        if coverage:
            print(
                "  coverage: unique="
                f"{coverage.get('unique_selected_neurons'):.2f}, "
                f"fraction={coverage.get('coverage_fraction'):.4f}"
            )
        curve = row.get("eval_curve") or []
        if curve:
            compact = [(int(x.get("step", 0)), round(float(x.get("nll_per_token", 0)), 5), x.get("gap_vs_dense")) for x in curve]
            print("  curve:", compact)


for name, path in continuation_reports.items():
    if Path(path).exists():
        summarize_continuation(Path(path))

if bench_output and Path(bench_output).exists():
    bench = _load_json(Path(bench_output))
    print("\n===", Path(bench_output).name, "===")
    for row in bench.get("rows", []):
        print(json.dumps({
            "size": f"{row['d_model']}x{row['d_ff']}x{row['tokens']}",
            "dense_fwd_bwd_ms": row.get("dense_forward_backward_ms"),
            "sparse_fwd_bwd_ms": row.get("sparse_forward_backward_ms"),
            "scatter_ms": row.get("grad_scatter_ms"),
            "speedup_no_scatter": row.get("forward_backward_speedup"),
            "speedup_with_scatter": row.get("forward_backward_scatter_speedup"),
            "peak_memory": row.get("peak_memory"),
        }, indent=2))



if 'union_eval_reports' in globals():
    for name, path in union_eval_reports.items():
        if Path(path).exists():
            report = _load_json(Path(path))
            print("\n===", Path(path).name, "===")
            rows = {row.get("variant"): row for row in report.get("rows", [])}
            dense = rows.get("dense", {})
            cluster = rows.get("static_cluster", {})
            for variant, union in rows.items():
                if not str(variant).startswith("global_union"):
                    continue
                print(json.dumps({
                    "variant": variant,
                    "dense_nll": dense.get("nll_per_token"),
                    "static_cluster_nll": cluster.get("nll_per_token"),
                    "union_nll": union.get("nll_per_token"),
                    "union_minus_cluster": (union.get("nll_per_token") - cluster.get("nll_per_token")) if union and cluster else None,
                    "union_minus_dense": (union.get("nll_per_token") - dense.get("nll_per_token")) if union and dense else None,
                    "active_cap": union.get("active_cap"),
                    "active_cap_by_layer": union.get("active_cap_by_layer"),
                    "avg_active_neurons": union.get("avg_active_neurons"),
                    "avg_active_fraction": union.get("avg_active_fraction"),
                    "union_recall": union.get("union_recall"),
                    "cluster_recall": cluster.get("candidate_recall"),
                }, indent=2))

if 'active_union_continuation_reports' in globals():
    active_dense_curve: dict[int, float] = {}
    for path in active_union_continuation_reports.values():
        if not Path(path).exists():
            continue
        report = _load_json(Path(path))
        dense_row = next((row for row in report.get("rows", []) if row.get("variant") == "dense_continuation"), None)
        if dense_row:
            active_dense_curve = {int(p.get("step", 0)): float(p.get("nll_per_token")) for p in dense_row.get("eval_curve", [])}
            break
    for name, path in active_union_continuation_reports.items():
        if not Path(path).exists():
            continue
        summarize_continuation(Path(path))
        if not active_dense_curve:
            continue
        report = _load_json(Path(path))
        for row in report.get("rows", []):
            if row.get("variant") == "dense_continuation":
                continue
            curve = row.get("eval_curve") or []
            if not curve:
                continue
            first = curve[0]
            last = curve[-1]
            init_gap = float(first.get("nll_per_token")) - active_dense_curve.get(int(first.get("step", 0)), float("nan"))
            final_gap = float(last.get("nll_per_token")) - active_dense_curve.get(int(last.get("step", 0)), float("nan"))
            print(f"  active-union gap vs shared dense curve: {init_gap:.6f} -> {final_gap:.6f}")

if 'active_union_bench_output' in globals() and active_union_bench_output and Path(active_union_bench_output).exists():
    bench = _load_json(Path(active_union_bench_output))
    print("\n===", Path(active_union_bench_output).name, "===")
    for row in bench.get("rows", []):
        print(json.dumps({
            "size": f"{row['d_model']}x{row['d_ff']}x{row['tokens']}",
            "active_m": row.get("active_m"),
            "active_fraction": row.get("active_fraction"),
            "dense_baseline": row.get("dense_baseline"),
            "swiglu_impl": row.get("swiglu_impl"),
            "triton_swiglu_backward_used": row.get("triton_swiglu_backward_used"),
            "indexed_fused_fwd_speedup": row.get("active_indexed_fused_forward_speedup", row.get("active_indexed_forward_speedup")),
            "indexed_fused_fwd_bwd_speedup": row.get("active_indexed_fused_forward_backward_speedup", row.get("active_indexed_forward_backward_speedup")),
            "packed_split_fwd_bwd_speedup": row.get("active_packed_split_forward_backward_speedup"),
            "packed_fused_fwd_speedup": row.get("active_packed_fused_forward_speedup", row.get("active_packed_forward_speedup")),
            "packed_fused_fwd_bwd_speedup": row.get("active_packed_fused_forward_backward_speedup", row.get("active_packed_forward_backward_speedup")),
            "dense_cuda_graph_ms": row.get("dense_cuda_graph_forward_backward_ms"),
            "packed_fused_cuda_graph_ms": row.get("active_packed_fused_cuda_graph_forward_backward_ms"),
            "packed_fused_cuda_graph_speedup": row.get("active_packed_fused_cuda_graph_forward_backward_speedup"),
            "dense_cuda_graph_error": row.get("dense_cuda_graph_error"),
            "packed_fused_cuda_graph_error": row.get("active_packed_fused_cuda_graph_forward_backward_error"),
            "max_indexed_packed_abs_diff": row.get("max_indexed_packed_abs_diff"),
            "max_split_fused_abs_diff": row.get("max_split_fused_abs_diff"),
        }, indent=2))

if 'full_model_bench_output' in globals() and full_model_bench_output and Path(full_model_bench_output).exists():
    bench = _load_json(Path(full_model_bench_output))
    print("\n===", Path(full_model_bench_output).name, "===")
    for row in bench.get("rows", []):
        coverage = ((row.get("coverage") or {}).get("mean") or {})
        print(json.dumps({
            "active_union_cap": row.get("active_union_cap"),
            "dtype": row.get("dtype"),
            "triton_swiglu_backward_used": row.get("triton_swiglu_backward_used"),
            "dense_total_step_ms": row.get("dense_total_step_ms"),
            "sparse_total_step_ms": row.get("sparse_total_step_ms"),
            "total_step_speedup": row.get("total_step_speedup"),
            "dense_fwd_bwd_ms": row.get("dense_forward_backward_ms"),
            "sparse_fwd_bwd_ms": row.get("sparse_forward_backward_ms"),
            "ffn_forward_speedup": row.get("ffn_forward_speedup"),
            "attention_forward_speedup": row.get("attention_forward_speedup"),
            "output_forward_speedup": row.get("output_forward_speedup"),
            "avg_active_neurons": coverage.get("unique_selected_neurons"),
            "avg_active_fraction": coverage.get("coverage_fraction"),
            "sparse_param_fraction": row.get("sparse_param_fraction"),
        }, indent=2))

print("\nRun root:", RUN_ROOT)
